In [12]:
import polars as pl
from extract import endpoints, get
from transform import transform_sports, transform_leagues, SportSchema, LeagueSchema, SportCollection
from pathlib import Path

In [13]:
data = Path('data')
tables = endpoints.keys()
paths = [data/(table+'.parquet') for table in tables]

In [14]:
for table, path in zip(tables, paths):
    if not path.exists():
        get(table).write_parquet(path)

In [15]:
sports = pl.scan_parquet(data/'sports.parquet')
leagues = pl.scan_parquet(data/'leagues.parquet')
divisions = pl.scan_parquet(data/'divisions.parquet')

In [16]:
sports = transform_sports(sports)
sports = SportSchema.validate(sports, cast=True)

In [17]:
leagues = transform_leagues(leagues)
leagues = LeagueSchema.validate(leagues, cast=True)

In [46]:
sports_collection, _ = SportCollection.filter(
    {
        'sports': sports,
        'leagues': leagues,
    }
)